# 19_Task9_Train — Train lại các bản tốt nhất trên data cũ + Dataset_bonus, so sánh và chọn model cuối

**Dữ liệu:** `manifest_all.csv` (305 video; `tap` cũ giữ nguyên, bonus chia ở `18_SplitBonus`). Test = 33 video cũ + 15 video bonus (tách riêng được).

**3 họ model × 4 seed (0–3) = 12 lần train**, tối đa **120 epoch, dừng sớm 20 epoch**, AdamW lr 1e-3, batch 16, CUDA:
- `tcn_v2`: 126 đặc trưng, augmentation y như `04_Trainv2` (chế độ `all`: xoay (10,25,20), nhiễu 0.01, lật gương).
- `wide45_lownoise`: 126 đặc trưng, xoay (40,45,40), nhiễu 0.005, lật gương (lấy từ cấu hình lưu trong checkpoint `wide45_lownoise`, xem `08_WebcamWide`).
- `step2`: 546 đặc trưng (thêm khoảng cách khớp), augmentation y như `13_Task8_Step2`. Ghép 4 seed của `step2` = **Step 3b** (`step2 ens4`).

**Ứng viên so sánh (15 dòng):** từng seed của mỗi họ (`<họ> s0..s3`) và ensemble 4 seed của mỗi họ (`<họ> ens4`).

**Đánh giá (cell 7–8):** cùng một test, góc xoay giả lập 0/15/30/45/60°, tiêu chí Decision (TB 5 ngưỡng 0.5→0.9), độ chính xác khi tin cậy ≥ 0.7, độ phủ, thời điểm đoán đúng đầu tiên, tốc độ suy luận. Bảng in một lượt, **model đứng đầu bảng** được ghi vào `final_model.json`. Cell 9–10: webcam, phím `m` đổi giữa các model đứng đầu.

**Checkpoint:** `Data_321/checkpoints_task9/<họ>/seed<k>/TCN.pt`, **không ghi đè** bản cũ. Có checkpoint đúng cấu hình thì tự nạp, không train lại (`FORCE_RETRAIN = True` ở cell 4 nếu muốn train lại). Checkpoint `tcn_v2` và `step2` đã train bằng 19 cũ vẫn dùng lại được vì cấu hình không đổi.

**Lưu ý:** augmentation của `wide45_lownoise` dựng theo cấu hình trong checkpoint + lật gương như 04; `07_TrainWide` chưa được đối chiếu, nếu khác thì sửa `augment_wide` ở cell 2.

**Cách chạy:** `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 10. Cần `manifest_all.csv` (chạy xong `18_SplitBonus.ipynb` trước).

In [1]:
#cell 1: nạp manifest_all.csv, dựng đặc trưng 126 cho train/val/test (CUDA), đánh dấu video test cũ / bonus
import math, time, json, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest_all.csv").exists(), "Chưa có manifest_all.csv, chạy 18_SplitBonus.ipynb trước"
m = pd.read_csv(BASE / "manifest_all.csv")
assert m.tap.isin(["train", "val", "test"]).all(), "manifest_all.csv có dòng chưa có tap"
Tmax, FPS_DATA = int(m.khung.max()), 120

def load_base_features(f):                                   # giống 13/15: cổ tay làm gốc, chia cỡ bàn tay, thêm vận tốc
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    pos = lm.reshape(len(lm), 63)
    has = ~np.isnan(pos[:, 0])
    vel = np.vstack([np.zeros((1, 63), np.float32), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32); Mk = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = load_base_features(f); X[i, :len(x)], Mk[i, :len(x)] = x, h
    return (torch.tensor(X, device=dev), torch.tensor(df.nhan.values, dtype=torch.long, device=dev), torch.tensor(Mk, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
Xte, yte, Mte = data["test"]

IS_BONUS = meta["test"].path.str.contains("Dataset_bonus", regex=False).to_numpy()      # video test thuộc Dataset_bonus
assert (~IS_BONUS).sum() == 33 and IS_BONUS.sum() == 15, f"Test cũ/bonus = {(~IS_BONUS).sum()}/{IS_BONUS.sum()}, mong đợi 33/15"
SUBIDX = {"all": np.ones(len(IS_BONUS), bool), "cu": ~IS_BONUS, "bonus": IS_BONUS}
SUBIDX = {k: torch.tensor(np.where(v)[0], device=dev) for k, v in SUBIDX.items()}

print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, Mk) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(Mk.sum().item()))
print(m.pivot_table(index="nguon", columns="tap", values="path", aggfunc="count")[["train", "val", "test"]])

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (214, 355, 126) | video: 214 | khung có tay: 26770
val (43, 355, 126) | video: 43 | khung có tay: 5190
test (48, 355, 126) | video: 48 | khung có tay: 5774
tap              train  val  test
nguon                            
Dataset_bonus       63   12    15
Dataset_hand        57   12    12
dataset_labeled     94   19    21


In [2]:
#cell 2: đặc trưng 546, xoay 3D, 3 kiểu augmentation (04 "all", wide45_lownoise, 13 step2), TCN, hàm loss
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def pos_vel(pos, hand_mask):                                 # (b,T,21,3) hoặc (b,T,63) -> 126 đặc trưng: tọa độ + vận tốc
    pos = pos.reshape(pos.shape[0], pos.shape[1], 63)
    vel = torch.zeros_like(pos); vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    return torch.cat([pos, vel], dim=-1)

def features_from_pos(pos, hand_mask):                       # giống 13/15: 126 gốc + 210 khoảng cách + 210 vận tốc khoảng cách
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist); dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_vel(pos, hand_mask), dist, dist_vel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

def rotate_deg(pos, ax, ay, az):                             # xoay 3D theo góc (độ) ax, ay, az, mỗi cái (b,1,1)
    ax, ay, az = [a * math.pi / 180 for a in (ax, ay, az)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    return torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

def augment_pos(X, hand_mask, rot_deg, noise_std):           # mỗi clip một góc xoay ngẫu nhiên + lật gương + nhiễu -> (b,T,21,3)
    b, T = X.shape[:2]
    pos = rotate_deg(X[..., :63].reshape(b, T, 21, 3).clone(), *[(torch.rand(b, 1, 1, device=X.device) * 2 - 1) * d for d in rot_deg])
    valid = hand_mask[:, :, None, None]
    flip = torch.rand(b, 1, 1, device=X.device) < 0.5
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])
    pos = pos + torch.randn_like(pos) * noise_std * valid
    return torch.where(valid, pos, torch.zeros_like(pos))

def augment_v2(X, hm):    return pos_vel(augment_pos(X, hm, (10, 25, 20), 0.01), hm)             # 04_Trainv2 "all"
def augment_wide(X, hm):  return pos_vel(augment_pos(X, hm, (40, 45, 40), 0.005), hm)            # wide45_lownoise
def augment_step2(X, hm): return features_from_pos(augment_pos(X, hm, (40, 45, 40), 0.005), hm)  # 13_Task8_Step2

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d; self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)]); self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

def loss_acc(net, X, y, Mk):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * Mk).sum() / Mk.sum().clamp(min=1), (correct & Mk).sum() / Mk.sum().clamp(min=1)

ID = lambda X, Mk: X
VARIANTS = {   # n_in: số đặc trưng | aug: augmentation lúc train | feat: đổi đặc trưng gốc 126 -> đầu vào model | cfg: dùng để nhận checkpoint cũ
    "tcn_v2":          dict(n_in=126, aug=augment_v2,    feat=ID,           cfg={"aug": "04_all", "n_in": 126}),
    "wide45_lownoise": dict(n_in=126, aug=augment_wide,  feat=ID,           cfg={"aug": "wide45_lownoise", "rot": (40, 45, 40), "noise": 0.005, "n_in": 126}),
    "step2":           dict(n_in=546, aug=augment_step2, feat=geo_from_base, cfg={"aug": "13_step2", "n_in": 546}),
}
Xv, yv, Mv = data["val"]
VAL = {k: v["feat"](Xv, Mv) for k, v in VARIANTS.items()}    # đặc trưng val dựng một lần
print("Đã định nghĩa:", list(VARIANTS), "| đầu vào:", {k: v["n_in"] for k, v in VARIANTS.items()})

Đã định nghĩa: ['tcn_v2', 'wide45_lownoise', 'step2'] | đầu vào: {'tcn_v2': 126, 'wide45_lownoise': 126, 'step2': 546}


In [3]:
#cell 3: cấu hình train và hàm train một model (tối đa 120 epoch, dừng sớm 20, lưu checkpoint tốt nhất theo val loss)
EPOCHS, PATIENCE, BS, SEEDS = 120, 20, 16, [0, 1, 2, 3]
CK = BASE / "checkpoints_task9"
ck_path = lambda name, seed: CK / name / f"seed{seed}" / "TCN.pt"

def train_one(name, seed, force=False):
    v, path = VARIANTS[name], ck_path(name, seed)
    cfg = {**v["cfg"], "variant": name, "seed": seed, "epochs": EPOCHS, "patience": PATIENCE,
           "n_train": len(data["train"][1]), "n_val": len(data["val"][1])}      # đổi dữ liệu/siêu tham số thì train lại
    if path.exists() and not force:
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            print(f"{name} seed {seed}: nạp checkpoint có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return
        print(f"{name} seed {seed}: checkpoint có cấu hình khác, sẽ train lại.")
    torch.manual_seed(seed)
    net = TCN(v["n_in"]).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, EPOCHS + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for s0 in range(0, len(Xt), BS):
            ids = perm[s0:s0 + BS]
            loss, _ = loss_acc(net, v["aug"](Xt[ids], Mt[ids]), yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, VAL[name], yv, Mv)
        hist.append({"epoch": ep, "train_loss": total / len(Xt), "val_loss": float(vl.item()), "val_acc": float(va.item())})
        if hist[-1]["val_loss"] < best:
            best, wait = hist[-1]["val_loss"], 0
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist,
                        "model": "TCN", "variant": f"task9_{name}_seed{seed}", "cfg": cfg}, path)
        else:
            wait += 1
        if ep % 10 == 0 or wait >= PATIENCE or ep == EPOCHS:
            print(f"{name} seed {seed} ep{ep:03d} | train {hist[-1]['train_loss']:.4f} | val {hist[-1]['val_loss']:.4f} | val_acc {hist[-1]['val_acc']:.3f}")
        if wait >= PATIENCE: break
    ck = torch.load(path, map_location=dev, weights_only=False)
    ck.update(stop_epoch=ep, seconds=time.time() - t0); torch.save(ck, path)   # ghi thêm epoch dừng và thời gian train
    print(f"{name} seed {seed}: tốt nhất epoch {ck['epoch']} | dừng ở epoch {ep} | val_loss {ck['val_loss']:.4f} | {ck['seconds']:.0f}s | đã lưu: {path}")
print("Checkpoint sẽ lưu ở:", CK)

Checkpoint sẽ lưu ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9


In [4]:
#cell 4: train 12 model (3 họ x 4 seed); có checkpoint đúng cấu hình thì tự nạp
FORCE_RETRAIN = False   # True chỉ khi muốn train lại tất cả
for name in VARIANTS:
    for seed in SEEDS:
        train_one(name, seed, FORCE_RETRAIN)
    print("-" * 80)
print("Xong. Checkpoint ở:", CK)

tcn_v2 seed 0: nạp checkpoint có sẵn | epoch 18 | val_loss 0.3333
tcn_v2 seed 1: nạp checkpoint có sẵn | epoch 21 | val_loss 0.3158
tcn_v2 seed 2: nạp checkpoint có sẵn | epoch 26 | val_loss 0.3485
tcn_v2 seed 3: nạp checkpoint có sẵn | epoch 28 | val_loss 0.3023
--------------------------------------------------------------------------------
wide45_lownoise seed 0: nạp checkpoint có sẵn | epoch 30 | val_loss 0.3251
wide45_lownoise seed 1: nạp checkpoint có sẵn | epoch 34 | val_loss 0.3544
wide45_lownoise seed 2: nạp checkpoint có sẵn | epoch 26 | val_loss 0.3391
wide45_lownoise seed 3: nạp checkpoint có sẵn | epoch 44 | val_loss 0.3468
--------------------------------------------------------------------------------
step2 seed 0: nạp checkpoint có sẵn | epoch 22 | val_loss 0.3331
step2 seed 1: nạp checkpoint có sẵn | epoch 27 | val_loss 0.3224
step2 seed 2: nạp checkpoint có sẵn | epoch 59 | val_loss 0.3151
step2 seed 3: nạp checkpoint có sẵn | epoch 20 | val_loss 0.3254
--------------

In [5]:
#cell 5: bảng tóm tắt 12 model trên tập val (nạp lại từ checkpoint) và lưu train_summary.csv
rows = []
for name, v in VARIANTS.items():
    for seed in SEEDS:
        p = ck_path(name, seed)
        if not p.exists(): print("Thiếu checkpoint:", p); continue
        ck = torch.load(p, map_location=dev, weights_only=False)
        net = TCN(v["n_in"]).to(dev); net.load_state_dict(ck["w"]); net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, VAL[name], yv, Mv)
        rows.append({"họ": name, "seed": seed, "epoch tốt nhất": ck["epoch"], "epoch dừng": ck.get("stop_epoch", np.nan),
                     "val_loss": vl.item(), "val_acc": va.item(), "giây": ck.get("seconds", np.nan)})
summary = pd.DataFrame(rows)
summary.to_csv(CK / "train_summary.csv", index=False)
display(summary.round(4))
display(summary.groupby("họ")[["val_loss", "val_acc"]].agg(["mean", "min", "max"]).round(4))
print("Đã lưu:", CK / "train_summary.csv")

,họ,seed,epoch tốt nhất,epoch dừng,val_loss,val_acc,giây
0,tcn_v2,0,18,38,0.3333,0.8692,2.7261
1,tcn_v2,1,21,41,0.3158,0.8753,2.3887
2,tcn_v2,2,26,46,0.3485,0.8536,3.2238
3,tcn_v2,3,28,48,0.3023,0.8840,3.5368
4,wide45_lownoise,0,30,50,0.3251,0.8713,3.9450
5,wide45_lownoise,1,34,54,0.3544,0.8730,3.7583
6,wide45_lownoise,2,26,46,0.3391,0.8464,2.9824
7,wide45_lownoise,3,44,64,0.3468,0.8565,3.8697
8,step2,0,22,42,0.3331,0.8761,3.0320
9,step2,1,27,47,0.3224,0.8651,3.3187


val_loss                 val_acc                
                    mean     min     max    mean     min     max
họ                                                              
step2             0.3240  0.3151  0.3331  0.8766  0.8651  0.8933
tcn_v2            0.3250  0.3023  0.3485  0.8705  0.8536  0.8840
wide45_lownoise   0.3414  0.3251  0.3544  0.8618  0.8464  0.8730

Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9\train_summary.csv


In [6]:
#cell 6: nạp 12 checkpoint, tạo danh sách ứng viên (từng seed + ensemble 4 seed của mỗi họ), hàm tính xác suất
NETS = {}
for fam, v in VARIANTS.items():
    for s in SEEDS:
        p = ck_path(fam, s); assert p.exists(), f"Thiếu {p}; chạy cell 4 trước"
        net = TCN(v["n_in"]).to(dev); net.load_state_dict(torch.load(p, map_location=dev, weights_only=False)["w"]); NETS[(fam, s)] = net.eval()

CANDS = {}
for fam in VARIANTS:
    CANDS.update({f"{fam} s{s}": [(fam, s)] for s in SEEDS})
    CANDS[f"{fam} ens{len(SEEDS)}"] = [(fam, s) for s in SEEDS]      # step2 ens4 = Step 3b

@torch.no_grad()
def last_probs(ks, X, Mk):                                   # xác suất trung bình của các net trong ks, mọi khung: (b,T,3)
    inp = {f: VARIANTS[f]["feat"](X, Mk) for f in {k[0] for k in ks}}
    return torch.stack([NETS[k](inp[k[0]]).softmax(-1) for k in ks]).mean(0)

@torch.no_grad()
def all_probs(X):                                            # xác suất của mọi ứng viên trên test X (đặc trưng và net chỉ tính một lần)
    inp = {f: v["feat"](X, Mte) for f, v in VARIANTS.items()}
    P1 = {k: n(inp[k[0]]).softmax(-1) for k, n in NETS.items()}
    return {name: torch.stack([P1[k] for k in ks]).mean(0) for name, ks in CANDS.items()}
print("Ứng viên:", len(CANDS), "|", list(CANDS))

Ứng viên: 15 | ['tcn_v2 s0', 'tcn_v2 s1', 'tcn_v2 s2', 'tcn_v2 s3', 'tcn_v2 ens4', 'wide45_lownoise s0', 'wide45_lownoise s1', 'wide45_lownoise s2', 'wide45_lownoise s3', 'wide45_lownoise ens4', 'step2 s0', 'step2 s1', 'step2 s2', 'step2 s3', 'step2 ens4']


In [7]:
#cell 7: chấm điểm cả 15 ứng viên trên cùng test: góc xoay giả lập 0/15/30/45/60, tiêu chí Decision (ngưỡng 0.5→0.9)
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]

def metrics(P, y, Mk):
    conf, pred = P.max(-1)
    ok = (pred == y[:, None]) & Mk
    r = {"acc_all": ok.sum().item() / max(Mk.sum().item(), 1) * 100}
    for t in THRS:
        k = int(round(t * 100)); hit = ok & (conf >= t); has = hit.any(1)
        r[f"v{k}"] = has.float().mean().item() * 100                                           # % video có khung đúng và đủ tin cậy
        r[f"ms{k}"] = (hit.float().argmax(1).float()[has].mean().item() / FPS_DATA * 1000) if has.any() else float("nan")
    r["tb"] = float(np.mean([r[f"v{int(round(t * 100))}"] for t in THRS]))
    sel = (conf >= 0.7) & Mk
    r["accc70"] = (ok & sel).sum().item() / max(sel.sum().item(), 1) * 100                      # độ chính xác khi tin cậy ≥ 0.7
    r["cov70"] = sel.sum().item() / max(Mk.sum().item(), 1) * 100                               # độ phủ ở ngưỡng 0.7
    return r

def rotated_test(mag, g):                                    # xoay mỗi video test một góc ngẫu nhiên trong ±mag độ (cả 3 trục)
    ang = lambda: (torch.rand(len(Xte), 1, 1, device=dev, generator=g) * 2 - 1) * mag
    pos = rotate_deg(Xte[..., :63].reshape(len(Xte), Tmax, 21, 3), ang(), ang(), ang())
    return pos_vel(torch.where(Mte[:, :, None, None], pos, torch.zeros_like(pos)), Mte)

MAGS, REPEAT = [0, 15, 30, 45, 60], 5
g = torch.Generator(device=dev).manual_seed(0)
rows = []
for mag in MAGS:
    acc = {}
    for _ in range(1 if mag == 0 else REPEAT):
        Xr = Xte if mag == 0 else rotated_test(mag, g)       # mọi ứng viên chấm trên cùng dữ liệu xoay
        for name, P in all_probs(Xr).items():
            for sub in (SUBIDX if mag == 0 else ["all"]):    # tách test cũ / bonus chỉ ở góc 0
                idx = SUBIDX[sub]
                acc.setdefault((name, sub), []).append(metrics(P[idx], yte[idx], Mte[idx]))
    for (name, sub), v in acc.items():
        rows.append({"model": name, "goc": mag, "tap": sub, **{k: float(np.nanmean([d[k] for d in v])) for k in v[0]}})
    print("xong góc", mag)
res = pd.DataFrame(rows)
res.to_csv(CK / "compare_task9.csv", index=False)
print("Đã lưu:", CK / "compare_task9.csv")

xong góc 0
xong góc 15
xong góc 30
xong góc 45
xong góc 60
Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9\compare_task9.csv


In [8]:
#cell 8: bảng so sánh một lượt, xếp hạng, chọn model đứng đầu và ghi final_model.json
A = res[res.tap == "all"]
at0 = lambda sub, col: res[(res.tap == sub) & (res.goc == 0)].set_index("model")[col]

xw, mw = Xte[:1, :128], Mte[:1, :128]
def bench(ks, n=100):                                        # ms cho một lần dự đoán (dựng đặc trưng + model), cửa sổ 128 khung như webcam
    for _ in range(10): last_probs(ks, xw, mw)
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n): last_probs(ks, xw, mw)
    torch.cuda.synchronize(); return (time.perf_counter() - t) / n * 1000

tab = pd.DataFrame({
    "TB 0° tất cả": at0("all", "tb"), "TB 0° cũ": at0("cu", "tb"), "TB 0° bonus": at0("bonus", "tb"),
    "TB mọi góc": A.groupby("model").tb.mean(),
    "acc 0°": at0("all", "acc_all"), "acc 45°": A[A.goc == 45].set_index("model").acc_all,
    "acc≥0.7": at0("all", "accc70"), "phủ 0.7": at0("all", "cov70"), "ms đoán đúng 0.7": at0("all", "ms70"),
    "ms/dự đoán": pd.Series({k: bench(ks) for k, ks in CANDS.items()}),
}).sort_values("TB mọi góc", ascending=False)
print("BẢNG SO SÁNH (test: tất cả = cũ 33 + bonus 15; xếp theo 'TB mọi góc' = TB 5 ngưỡng, trung bình 5 góc):")
display(tab.round(1))

print("\nTB 5 ngưỡng theo góc (%)");   display(A.pivot_table(index="model", columns="goc", values="tb").reindex(tab.index).round(1))
print("Độ chính xác mọi khung theo góc (%)"); display(A.pivot_table(index="model", columns="goc", values="acc_all").reindex(tab.index).round(1))

single = tab[~tab.index.str.contains("ens")]
print("Độ lệch giữa các seed của cùng một họ (TB mọi góc):")
display(single.groupby([n.rsplit(" ", 1)[0] for n in single.index])["TB mọi góc"].agg(["mean", "min", "max"]).round(1))

BEST = tab.index[0]
final = {"model": BEST, "checkpoints": [str(ck_path(*k)) for k in CANDS[BEST]],
         "n_in": [VARIANTS[k[0]]["n_in"] for k in CANDS[BEST]], "TB_moi_goc": float(tab.loc[BEST, "TB mọi góc"])}
(CK / "final_model.json").write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nĐỨNG ĐẦU BẢNG: {BEST} | TB mọi góc {final['TB_moi_goc']:.1f} | đã ghi {CK / 'final_model.json'}")
print("Lưu ý: tiêu chí chỉ tham khảo; chênh nhỏ hơn độ lệch giữa các seed (bảng trên) hoặc test bonus chỉ 15 video thì chưa chắc chắn. Xác nhận bằng webcam (cell 10).")

BẢNG SO SÁNH (test: tất cả = cũ 33 + bonus 15; xếp theo 'TB mọi góc' = TB 5 ngưỡng, trung bình 5 góc):


,TB 0° tất cả,TB 0° cũ,TB 0° bonus,TB mọi góc,acc 0°,acc 45°,acc≥0.7,phủ 0.7,ms đoán đúng 0.7,ms/dự đoán
step2 s1,97.1,98.8,93.3,96.7,85.9,86.6,95.8,81.8,178.4,1.0
step2 s3,96.7,98.8,92.0,96.5,84.7,84.8,93.1,83.1,177.7,1.1
step2 s2,96.3,99.4,89.3,96.2,87.7,86.5,95.1,80.6,176.1,1.0
step2 ens4,95.4,98.2,89.3,95.6,86.6,86.8,95.6,81.3,182.4,2.9
step2 s0,94.6,97.0,89.3,94.6,86.2,85.9,97.0,77.8,178.6,1.0
wide45_lownoise s1,94.6,97.6,88.0,91.6,83.3,82.3,94.1,78.8,213.3,0.6
wide45_lownoise s0,93.3,97.0,85.3,91.0,83.3,82.6,92.0,81.4,228.2,0.6
wide45_lownoise s3,90.8,97.0,77.3,89.7,83.3,82.9,96.5,75.6,205.2,0.6
wide45_lownoise ens4,91.7,95.8,82.7,89.5,84.0,83.4,96.1,76.3,236.5,2.3
tcn_v2 s2,92.9,97.6,82.7,88.8,83.5,78.9,94.0,76.2,209.6,0.6



TB 5 ngưỡng theo góc (%)


goc,0,15,30,45,60
step2 s1,97.1,96.8,96.7,96.8,96.3
step2 s3,96.7,96.8,96.7,96.2,96.2
step2 s2,96.3,96.4,96.2,96.2,96.0
step2 ens4,95.4,95.5,95.6,95.8,95.5
step2 s0,94.6,94.4,94.8,94.9,94.5
wide45_lownoise s1,94.6,93.8,91.2,90.0,88.2
wide45_lownoise s0,93.3,93.4,91.7,89.4,86.9
wide45_lownoise s3,90.8,90.8,90.4,89.0,87.4
wide45_lownoise ens4,91.7,91.8,89.8,88.3,86.0
tcn_v2 s2,92.9,93.2,91.5,84.6,82.0


Độ chính xác mọi khung theo góc (%)


goc,0,15,30,45,60
step2 s1,85.9,85.8,85.9,86.6,86.0
step2 s3,84.7,84.7,85.1,84.8,84.7
step2 s2,87.7,87.5,86.9,86.5,87.1
step2 ens4,86.6,86.6,86.8,86.8,86.5
step2 s0,86.2,86.1,86.6,85.9,85.7
wide45_lownoise s1,83.3,83.3,82.6,82.3,80.3
wide45_lownoise s0,83.3,83.9,82.6,82.6,79.7
wide45_lownoise s3,83.3,82.9,83.6,82.9,81.2
wide45_lownoise ens4,84.0,84.0,83.4,83.4,81.8
tcn_v2 s2,83.5,83.2,83.1,78.9,76.2


Độ lệch giữa các seed của cùng một họ (TB mọi góc):


,mean,min,max
step2,96.0,94.6,96.7
tcn_v2,85.1,81.8,88.8
wide45_lownoise,89.9,87.6,91.6



ĐỨNG ĐẦU BẢNG: step2 s1 | TB mọi góc 96.7 | đã ghi C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9\final_model.json
Lưu ý: tiêu chí chỉ tham khảo; chênh nhỏ hơn độ lệch giữa các seed (bảng trên) hoặc test bonus chỉ 15 video thì chưa chắc chắn. Xác nhận bằng webcam (cell 10).


In [9]:
#cell 9: chuẩn bị MediaPipe, nội suy webcam -> 120 fps, giao diện (giống 08_WebcamWide), chọn các model để đổi bằng phím m
import cv2
import mediapipe as mp

WEB_N = 4                                                    # số model đứng đầu bảng để đổi bằng phím m (hoặc gán WEB = [...] tên ứng viên)
WEB = list(tab.index[:WEB_N])
TASK = BASE / "hand_landmarker.task"
assert TASK.exists(), f"Không thấy {TASK}"
CLASSES = ["paper", "rock", "scissors"]
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}    # BGR: paper = đỏ, rock = xanh dương, scissors = xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()     # timestamp của MediaPipe phải luôn tăng, kể cả khi chạy lại cell 10

def detect(frame_bgr, ts_ms):                                # tọa độ ảnh chuẩn hóa (0-1), đúng loại dùng lúc train (đã kiểm ở 08)
    res_ = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)), ts_ms)
    if not res_.hand_landmarks: return None
    return np.array([[p.x, p.y, p.z] for p in res_.hand_landmarks[0]], np.float32)

WIN = 128
buf = collections.deque(maxlen=WIN); state = {"prev_lm": None, "prev_t": None}
def push(lm, t):                                             # thêm khung webcam, nội suy tuyến tính lên 120 fps
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_DATA))
    if lm is None: buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None: buf.append(lm)
    else: buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def webcam_base_features(lm):                                # (T,21,3) có thể chứa NaN -> 126 đặc trưng gốc, giống lúc train
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    q = np.nan_to_num(q / np.maximum(np.linalg.norm(q[:, 9], axis=1)[:, None, None], 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63); vel = np.zeros_like(pos)
    vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def predict(name):
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]): return None
    base, has = webcam_base_features(lm)
    P = last_probs(CANDS[name], torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev))[0, -1]
    conf, c = P.max(0)
    return int(c), float(conf)

def put_label(img, text, x, y, bg, scale=0.8):               # chữ đen trên hộp màu; (x, y) là góc dưới-trái
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, lm, color, label, mirror=True):           # khung xương + khung bao + nhãn cùng màu
    h, w = img.shape[:2]
    px, py = (lm[:, 0] * w).astype(int), (lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES: cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 3, cv2.LINE_AA)
    for i in range(21):
        cv2.circle(img, (px[i], py[i]), 6, color, -1, cv2.LINE_AA); cv2.circle(img, (px[i], py[i]), 6, BLACK, 1, cv2.LINE_AA)
    x0, y0, x1, y1 = px.min() - 15, py.min() - 15, px.max() + 15, py.max() + 15
    cv2.rectangle(img, (x0, y0), (x1, y1), color, 3, cv2.LINE_AA)
    ly = y1 + 34 if (x0 < 330 and y0 < 160) else y0           # tránh đè lên khối FPS/model ở góc trên trái
    put_label(img, label, max(x0, 0), min(max(ly, 32), h - 5), color)

def draw_hud(img, fps, thr, name):
    put_label(img, f"FPS {fps:.0f} | Nguong >= {thr:.2f}", 10, 35, YELLOW, 0.6)
    put_label(img, f"MODEL: {name}", 10, 70, YELLOW, 0.6)
    put_label(img, "m: doi model | [ ]: nguong | q: thoat", 10, 105, YELLOW, 0.5)
    for i, n in enumerate(CLASSES): put_label(img, n, img.shape[1] - 125, img.shape[0] - 10 - 40 * (2 - i), COLORS[i])
print("Model webcam (phím m đổi):", WEB)

Model webcam (phím m đổi): ['step2 s1', 'step2 s3', 'step2 s2', 'step2 ens4']


In [11]:
#cell 10: chạy webcam — q: thoát | m: đổi model | [ ]: giảm/tăng ngưỡng tin cậy; nhìn FPS để biết ensemble có chậm không
CAM_ID, MIRROR, THR_CAM, WHICH = 0, True, 0.70, 0
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok: break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))           # MediaPipe nhận ảnh gốc (không lật), giống lúc trích landmark
        push(lm, now)
        name = WEB[WHICH]
        out = predict(name) if lm is not None else None
        disp = cv2.flip(frame, 1) if MIRROR else frame
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        if out is None:
            put_label(disp, "no hand", 10, 145, GRAY)
        else:
            c, conf = out; sure = conf >= THR_CAM
            draw_hand(disp, lm, COLORS[c] if sure else GRAY, f"{CLASSES[c]} {conf:.0%}" if sure else f"{CLASSES[c]}? {conf:.0%}", MIRROR)
        draw_hud(disp, fps, THR_CAM, name)
        cv2.imshow("Task 9 - RPS webcam (q: thoat)", disp)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == ord("m"): WHICH = (WHICH + 1) % len(WEB)
        if k == ord("["): THR_CAM = max(0.05, THR_CAM - 0.05)
        if k == ord("]"): THR_CAM = min(0.95, THR_CAM + 0.05)
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")

Đã đóng webcam.
